# Project 2 Part I (Core)

*Project 2: Initial Analysis and Problem Selection*

---

## Objective

Perform an initial Exploratory Data Analysis (EDA) on at least four datasets, identify and diagnose potential issues, and select a specific problem to address (regression, classification, clustering, or forecasting). Submit a repository containing the selected dataset, the initial EDA, and a description of the chosen problem.

---

# Part I: Dataset Search and Analysis

## Problem Statement

Develop a forecasting model to predict future temperature patterns and support tourism planning by identifying the most favorable times of the year to visit different locations.

# Dataset Description

This dataset contains daily temperature records collected during 2014 from meteorological stations operated by the Chilean Dirección General de Aeronáutica Civil (DGAC). The stations are located at airports, airfields, and other observation sites across Chile, including Antarctica. The dataset includes the observation location, month, day, minimum temperature, and maximum temperature.

## Dataset Source

In this project, we analyze the Daily Temperatures in Chile dataset, obtained from the Chilean Government Open Data Portal. The data was originally published by the Dirección General de Aeronáutica Civil (DGAC) and contains daily temperature observations recorded at meteorological stations throughout Chile during 2014.

The dataset can be accessed and downloaded from the following link:

https://datos.gob.cl/dataset/2725
https://datos.gob.cl/uploads/recursos/temperatura2014-csv.csv

## Dataset Information

- Number of observations: **11870**
- Number of features: **5**
- Target variable: 
    - **Primary**: `max_temp`
    - **Secondary mention**: `min_temp` (optional)

## Data Dictionary

## Data Dictionary

| Column | Data Type | Description | Example |
|----------|----------|-------------|---------|
| `station` | String | Name of the meteorological station where the temperature observation was recorded. | `Chacalluta, Arica Ap.` |
| `month`    | Integer   | Month of the observation, represented as a number from 1 to 12.                                       | `1` (January)           |
| `day`      | Integer   | Day of the month when the observation was recorded.                                                   | `15`                    |
| `min_temp` | Float     | Minimum temperature recorded for the location on the specified day, measured in degrees Celsius (°C). | `18.8`                  |
| `max_temp` | Float     | Maximum temperature recorded for the location on the specified day, measured in degrees Celsius (°C). | `25.6`                  |

### Example Record

| station                  | month | day | min_temp | max_temp |
| --------------------- | ----- | --- | -------- | -------- |
| Chacalluta, Arica Ap. | 1     | 2   | 18.8     | 25.6     |

**Record Interpretation:** On January 2, 2014, the weather station *Chacalluta, Arica Ap.* recorded a minimum temperature of **18.8°C** and a maximum temperature of **25.6°C**.

---

# Data Loading and Inspection

## Import Libraries

In [57]:
# Standard library imports
import math
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

### Dataset Availability and Path Setup

In [58]:
data_dir = Path("../datasets")

# List available files (for verification)
available_files = list(data_dir.iterdir())
print("Available files:", available_files)
file_name = "4_temp_chile_2014.csv"
dataset_path = data_dir / file_name

Available files: [WindowsPath('../datasets/1_aircrafts_chile_30042026.csv'), WindowsPath('../datasets/2_earthquakes_chile.csv'), WindowsPath('../datasets/3_healthy_diet.csv'), WindowsPath('../datasets/4_temp_chile_2014.csv'), WindowsPath('../datasets/README.md')]


In [59]:
# Validate dataset existence
if not dataset_path.exists():
    raise FileNotFoundError(f"{file_name} not found in {data_dir}")

print("Dataset found at:", dataset_path)

Dataset found at: ..\datasets\4_temp_chile_2014.csv


## Load Dataset

In [60]:
columns = ['station', 'month', 'day', 'min_temp',
        'max_temp']

df = pd.read_csv(dataset_path, sep=";", names=columns, header=0)

## Dataset Overview


### First and Last Records

In [61]:
df.head()

,station,month,day,min_temp,max_temp
0,"Chacalluta, Arica Ap.",1,2,18.8,25.6
1,"Chacalluta, Arica Ap.",1,3,18.3,25.1
2,"Chacalluta, Arica Ap.",1,4,19.2,25.6
3,"Chacalluta, Arica Ap.",1,5,20.5,26.1
4,"Chacalluta, Arica Ap.",1,6,20.2,25.3


In [62]:
df.tail()

,station,month,day,min_temp,max_temp
11865,"C.M.A. Eduardo Frei Montalva, Antártica",12,27,-2.2,-0.2
11866,"C.M.A. Eduardo Frei Montalva, Antártica",12,28,-1.3,0.7
11867,"C.M.A. Eduardo Frei Montalva, Antártica",12,29,-0.7,1.2
11868,"C.M.A. Eduardo Frei Montalva, Antártica",12,30,-1.0,1.1
11869,"C.M.A. Eduardo Frei Montalva, Antártica",12,31,-0.7,1.0


### Verify Structure

In [63]:
print(f"{file_name}")
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} rows, {df.shape[1]} columns\n")

4_temp_chile_2014.csv
Shape: (11870, 5)
→ 11870 rows, 5 columns



---

## Initial Data Inspection

### Inspect Data Types

In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11870 entries, 0 to 11869
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   station   11870 non-null  str    
 1   month     11870 non-null  int64  
 2   day       11870 non-null  int64  
 3   min_temp  11835 non-null  float64
 4   max_temp  11779 non-null  float64
dtypes: float64(2), int64(2), str(1)
memory usage: 463.8 KB


The dataset contains 11,870 observations and 5 variables.

The `station` column is categorical, while `month` and `day` are integer variables representing temporal information. The `min_temp` and `max_temp` columns are numeric and contain missing values.

No complete date field is available in the dataset. Instead, temporal information is provided through separate `month` and `day` columns, which may be combined later if required for forecasting or seasonal analysis.

The dataset contains observations from a single year (2014), which may limit the ability to model long-term temperature trends. However, the data remains suitable for analyzing seasonal patterns and short-term temperature behavior across locations.

In [65]:
df.columns.tolist()

['station', 'month', 'day', 'min_temp', 'max_temp']

### Descriptive Statistics

In [66]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
month,11870.0,6.479781,3.443619,1.0,4.0,6.0,9.0,12.0
day,11870.0,15.712721,8.794776,1.0,8.0,16.0,23.0,31.0
min_temp,11835.0,7.284875,5.894082,-17.4,3.2,7.3,11.4,22.8
max_temp,11779.0,16.812174,7.161134,-10.1,12.4,17.0,21.8,39.2


The dataset contains observations from all 12 months of the year.

The `day` variable ranges from 1 to 31, indicating that observations were recorded throughout the entire month cycle.

Minimum temperatures range from -17.4°C to 22.8°C, reflecting the broad geographic coverage of the dataset, which includes locations from northern Chile to Antarctica.

Maximum temperatures range from -10.1°C to 39.2°C, highlighting the wide variety of climatic conditions represented across the observation stations.

The median minimum temperature is 7.3°C, while the median maximum temperature is 17.0°C, suggesting generally mild temperature conditions across the stations included in the dataset.

In [67]:
df.describe(include="str").T

,count,unique,top,freq
station,11870,33,Diego Aracena Iquique Ap.,365


The dataset contains records from 33 unique observation stations distributed across Chile.

Multiple stations have 365 observations, suggesting that daily temperature measurements were consistently recorded throughout the year at those locations.

---

### Missing Values

In [80]:
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

missing_summary.round(2)

,missing_count,missing_percent
station,0,0.00
month,0,0.00
day,0,0.00
min_temp,35,0.29
max_temp,91,0.77


Less than 1% of values are missing in each temperature column (0.29% in `min_temp` and 0.77% in `max_temp`), so the amount of missing data is very small.

Both temperature columns have nulls, they could be filled with knn regressor or similar impute, as temperature is a continue measure and does not change abruptly. Both `min_temp` and `max_temp` contain missing values, with more gaps in `max_temp`.

Since temperature is a continuous variable and usually varies smoothly over time, these values can be reasonably imputed using methods that preserve local patterns, such as KNN imputation or interpolation, rather than simple mean/median filling.

### Duplicate Records Check

In [97]:
n_duplicated = df.duplicated().sum()

duplicate_pct = (
    n_duplicated / len(df) * 100
)

n_duplicated

np.int64(0)

The dataset contains no duplicate records.

---

## Distribution Analysis

### Numeric

#### Histograms

In [70]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

# Dashboard layout
n_cols = 4
n_rows = math.ceil(len(numeric_columns) / n_cols)

# Create subplot figure
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=numeric_columns
)

# Add histograms
for i, col in enumerate(numeric_columns):
    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Histogram(
            x=df[col],
            name=col,
            nbinsx=30
        ),
        row=row,
        col=col_pos
    )

# Update layout
fig.update_layout(
    title="Interactive Histogram Dashboard — Numeric Features",
    height=300 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white",
    bargap=0.05
)

# Descriptive axis labels
fig.update_xaxes(title_text="Feature Values")
fig.update_yaxes(title_text="Number of Observations")

# Show dashboard
fig.show()

**Analysis**

**Temporal Features (`month`, `day`)**

Both `month` and `day` show near-uniform distributions, indicating the data is evenly sampled across time with no noticeable collection bias.

**Temperature Variables**

*Minimum Temperature (`min_temp`)*

The distribution of `min_temp` is approximately bell-shaped with slight skewness toward lower values. It is centered around ~7°C, with occasional cold extremes below -10°C.

*Maximum Temperature (`max_temp`)*

`max_temp` is also approximately normal and largely symmetric around ~16°C, with mild tail behavior on both ends rather than a clear directional skew.

**Key Insights**

- Temporal variables are evenly distributed, supporting unbiased time coverage.
- Temperature variables are approximately normally distributed, with mild skewness reflecting natural climate variability.
- No strong asymmetry or abnormal distribution patterns are observed, suggesting good data quality.

### Categorical

#### Categorical Value Consistency

In [71]:
categorical_cols = df.select_dtypes(
    include=["str", "category"]
).columns

for col in categorical_cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"{'='*40}")

    summary = (
        df[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    summary.columns = [col, 'Count']

    summary['Percentage'] = (
        summary['Count'] / len(df) * 100
    ).round(2)

    print(summary)


Column: station
                                     station  Count  Percentage
0                  Diego Aracena Iquique Ap.    365        3.07
1                         El Loa, Calama Ad.    365        3.07
2             Cerro Moreno  Antofagasta  Ap.    365        3.07
3               Mataveri  Isla de Pascua Ap.    365        3.07
4          Desierto de Atacama, Caldera  Ad.    365        3.07
5                  La Florida, La Serena Ad.    365        3.07
6                             Rodelillo, Ad.    365        3.07
7              Eulógio Sánchez, Tobalaba Ad.    365        3.07
8                    Quinta Normal, Santiago    365        3.07
9                         Pudahuel Santiago     365        3.07
10                        Santo Domingo, Ad.    365        3.07
11   Juan Fernández, Estación Meteorológica.    365        3.07
12                General Freire, Curicó Ad.    365        3.07
13   General Bernardo O'Higgins, Chillán Ad.    365        3.07
14                  Car

In [78]:
df["station"].nunique()
sorted(df["station"].unique())[:20]

['Alto Palena Ad.',
 'Balmaceda Ad.',
 'C.M.A. Eduardo Frei Montalva, Antártica ',
 'Carlos Ibañez, Punta Arenas Ap.',
 'Carriel Sur, Concepción.',
 'Cañal Bajo,  Osorno Ad.',
 'Cerro Moreno  Antofagasta  Ap.',
 'Chacalluta, Arica Ap.',
 'Chile Chico Ad.',
 'Desierto de Atacama, Caldera  Ad.',
 'Diego Aracena Iquique Ap.',
 'El Loa, Calama Ad.',
 'El Tepual  Puerto Montt Ap.',
 'Eulógio Sánchez, Tobalaba Ad.',
 'Fuentes Martínez, Porvenir Ad.',
 'Futaleufú Ad.',
 "General Bernardo O'Higgins, Chillán Ad.",
 'General Freire, Curicó Ad.',
 'Guardia Marina Zañartu, Pto Williams Ad.',
 'Juan Fernández, Estación Meteorológica.']

In [72]:
MAX_CATEGORIES = 10

n_cols = 2
n_rows = math.ceil(len(categorical_cols) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=categorical_cols,
    vertical_spacing=0.10
)

for i, col in enumerate(categorical_cols):

    summary = (
        df[col]
        .value_counts(dropna=False)
        .head(MAX_CATEGORIES)
        .reset_index()
    )

    summary.columns = [col, "Count"]

    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Bar(
            x=summary[col].astype(str),
            y=summary["Count"],
            text=summary["Count"],
            textposition="outside",
            name=col
        ),
        row=row,
        col=col_pos
    )

fig.update_layout(
    title="Categorical Variables Distribution Dashboard",
    height=400 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white"
)

fig.show()

**Station Distribution**

The dataset contains measurements from **multiple meteorological stations across Chile**, with a total of `33` distinct locations.

Each station contributes approximately the same number of observations (around 363–365 records), indicating a well-balanced dataset with minimal sampling imbalance across locations.

Small deviations (e.g., 351 or 209 records for a few stations) likely reflect minor data loss or incomplete time coverage.

**Value Consistency**

All station categories are valid and consistently formatted, with no unexpected or missing labels. Minor variations in naming (e.g., spacing differences) are present but do not affect interpretation.

**Geographical Coverage Insight**

Stations are distributed across a wide geographic range, including:

- Northern Chile (e.g., Arica, Iquique, Calama)
- Central regions (e.g., Santiago, Concepción, Chillán)
- Southern regions (e.g., Valdivia, Punta Arenas, Puerto Williams)
- Remote and special zones (e.g., Juan Fernández, Antarctic stations)

This broad coverage ensures the dataset captures diverse climatic conditions across the country, improving its representativeness for environmental or climate analysis.

**Key Insights**

- Station data is nearly balanced, supporting unbiased regional comparisons.
- Geographic coverage is extensive, spanning mainland Chile, islands, and Antarctica.
- No missing or inconsistent categorical labels are observed in `station`.

### Countplots

#### Outlier Detection

In [73]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

if len(numeric_columns) > 0:

    # create boxplots
    # Dashboard layout
    n_cols = 4
    n_rows = math.ceil(len(numeric_columns) / n_cols)

    # Create subplot figure
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=numeric_columns
    )

    # Add boxplots
    for i, col in enumerate(numeric_columns):
        row = (i // n_cols) + 1
        col_pos = (i % n_cols) + 1

        fig.add_trace(
            go.Box(
                y=df[col],
                name=col,
                boxmean=True
            ),
            row=row,
            col=col_pos
        )

    # Update layout
    fig.update_layout(
        title="Interactive Boxplot Dashboard — Numeric Features",
        height=300 * n_rows,
        width=1400,
        showlegend=False,
        template="plotly_white"
    )

    # Add axis labels
    fig.update_yaxes(title_text="Observed Values")

    # Show dashboard
    fig.show()

else:
    print("No numeric variables available for outlier analysis.")

**Analysis**

Outliers are only present in the temperature variables (`min_temp` and `max_temp`). However, the proportion is low (<1% in both cases), indicating limited extreme values.

Importantly, these outliers are likely **valid environmental observations rather than data errors**, since temperature naturally includes occasional extreme conditions.

Outliers are present but likely reflect valid environmental extremes and are retained for analysis.

**Key Insights**

- No outliers detected in temporal features (`month`, `day`), as expected.
- Temperature variables contain a small proportion of extreme values, consistent with real-world variability.
- Overall data quality is strong, with no indication of erroneous outlier behavior.

IQR calculation.

In [74]:
feature_columns = df.select_dtypes(
    include=np.number
).columns

outlier_summary = []

for col in feature_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_summary.append({
        "Feature": col,
        "Outliers": n_outliers,
        "Percentage": round(n_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

,Feature,Outliers,Percentage
0,month,0,0.00
1,day,0,0.00
2,min_temp,49,0.41
3,max_temp,95,0.80


---

# Exploratory Data Analysis (EDA)

The goal of this section is to understand data quality, structure, and underlying climate patterns before modeling temperature forecasting.

### Correlation Analysis

In [75]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

corr_matrix = df[numeric_columns].corr()

corr_matrix

,month,day,min_temp,max_temp
month,1.000000,0.011868,-0.108030,-0.143611
day,0.011868,1.000000,-0.001152,-0.017469
min_temp,-0.108030,-0.001152,1.000000,0.656876
max_temp,-0.143611,-0.017469,0.656876,1.000000


### Heatmap visualization

In [76]:
# Correlation matrix
corr_matrix = df[numeric_columns].corr()

# Create mask for upper triangle
mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool)
)

# Apply mask
corr_matrix_masked = corr_matrix.mask(mask)

fig = px.imshow(
    corr_matrix_masked,
    text_auto=".2f",
    color_continuous_scale="BuGn",
    aspect="auto",
    title="Correlation Matrix Heatmap"
)

fig.update_layout(
    width=900,
    height=800
)

fig.show()

**Heatmap Interpretation**

The strongest relationship is observed between `min_temp` and `max_temp`, with a moderate positive correlation of **0.66**. This indicates that higher minimum temperatures are generally associated with higher maximum temperatures, as expected in daily climate patterns.

All other correlations are close to zero, suggesting:

- No meaningful linear relationship between temperature variables and time features (`month`, `day`)
- `month` and `day` do not strongly influence individual temperature values in a linear sense

**Key Insights**

- Temperature variables are moderately correlated, reflecting shared daily climate conditions.
- Temporal variables (`month`, `day`) show negligible correlation with temperature, suggesting that seasonal patterns are likely nonlinear or require more complex modeling.
- No signs of multicollinearity issues beyond the expected relationship between min/max temperature.

## Seasonality Analysis

In [81]:
# Group by month and compute mean temperatures
monthly_trends = df.groupby("month")[["min_temp", "max_temp"]].mean().reset_index()

monthly_trends

,month,min_temp,max_temp
0,1,10.913894,22.056654
1,2,10.213109,21.967316
2,3,8.855839,19.640668
3,4,6.846450,17.159898
4,5,6.210552,14.325173
5,6,4.384302,11.562398
6,7,4.010914,11.554402
7,8,5.263479,13.475873
8,9,5.922571,14.035782
9,10,7.084848,17.341327


In [82]:
fig = go.Figure()

# Min temperature
fig.add_trace(go.Scatter(
    x=monthly_trends["month"],
    y=monthly_trends["min_temp"],
    mode="lines+markers",
    name="Min Temp"
))

# Max temperature
fig.add_trace(go.Scatter(
    x=monthly_trends["month"],
    y=monthly_trends["max_temp"],
    mode="lines+markers",
    name="Max Temp"
))

fig.update_layout(
    title="Monthly Temperature Trends",
    xaxis_title="Month",
    yaxis_title="Temperature (°C)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

**Monthly Temperature Patterns**

Both minimum and maximum temperatures show a clear seasonal pattern across the year.

- Temperatures are higher during the summer months and lower during winter months, indicating a strong annual cycle.
- `max_temp` follows the same seasonal trend as `min_temp`, but with greater amplitude, reflecting higher daily variability in daytime heating.
- The difference between `min_temp` and `max_temp` remains relatively consistent throughout the year.

**Key Insights**

- Clear **seasonality is present**, confirming expected climate-driven behavior.
- Temperature variation is strongly driven by **time of year (month)** rather than day-level variation.
- Seasonal structure suggests that **time-based features (e.g., month or seasonal encoding)** would be valuable in predictive models.

## Station Climate Ranking

In [87]:
station_climate = df.groupby("station").agg(
    mean_min_temp=("min_temp", "mean"),
    mean_max_temp=("max_temp", "mean"),
)

station_climate["mean_temp"] = (
    station_climate["mean_min_temp"] + station_climate["mean_max_temp"]
) / 2

station_climate["temp_variability"] = (
    station_climate["mean_max_temp"] - station_climate["mean_min_temp"]
)

station_climate.head()

,mean_min_temp,mean_max_temp,mean_temp,temp_variability
station,,,,
Alto Palena Ad.,5.801950,15.131180,10.466565,9.329230
Balmaceda Ad.,2.422740,12.351507,7.387123,9.928767
"C.M.A. Eduardo Frei Montalva, Antártica",-4.292603,-0.675616,-2.484110,3.616986
"Carlos Ibañez, Punta Arenas Ap.",3.081370,9.732329,6.406849,6.650959
"Carriel Sur, Concepción.",8.944384,17.886849,13.415616,8.942466


In [88]:
coldest = station_climate.sort_values("mean_temp").head(10)
coldest

,mean_min_temp,mean_max_temp,mean_temp,temp_variability
station,,,,
"C.M.A. Eduardo Frei Montalva, Antártica",-4.292603,-0.675616,-2.484110,3.616986
"Guardia Marina Zañartu, Pto Williams Ad.",2.412363,9.099726,5.756044,6.687363
"Carlos Ibañez, Punta Arenas Ap.",3.081370,9.732329,6.406849,6.650959
"Teniente Gallardo, Puerto Natales Ad.",2.066761,11.047293,6.557027,8.980532
"Fuentes Martínez, Porvenir Ad.",3.211730,10.473244,6.842487,7.261514
Balmaceda Ad.,2.422740,12.351507,7.387123,9.928767
Lord Cochrane Ad.,3.440385,13.761644,8.601014,10.321259
"Teniente Vidal, Coyhaique Ad.",4.557808,13.665753,9.111781,9.107945
Chile Chico Ad.,3.228767,15.519780,9.374274,12.291013


In [89]:
warmest = station_climate.sort_values("mean_temp", ascending=False).head(10)
warmest

,mean_min_temp,mean_max_temp,mean_temp,temp_variability
station,,,,
Mataveri Isla de Pascua Ap.,17.435342,23.404932,20.420137,5.969589
"Chacalluta, Arica Ap.",16.934066,21.478022,19.206044,4.543956
Diego Aracena Iquique Ap.,16.088493,20.776712,18.432603,4.688219
Cerro Moreno Antofagasta Ap.,14.237534,19.283562,16.760548,5.046027
"Quinta Normal, Santiago",8.536438,23.550137,16.043288,15.013699
"Eulógio Sánchez, Tobalaba Ad.",8.845753,22.845429,15.845591,13.999676
"Desierto de Atacama, Caldera Ad.",11.561644,19.619178,15.590411,8.057534
Pudahuel Santiago,8.192877,22.926575,15.559726,14.733699
"Juan Fernández, Estación Meteorológica.",13.012877,18.010959,15.511918,4.998082


In [90]:
most_variable = station_climate.sort_values("temp_variability", ascending=False).head(10)
most_variable

,mean_min_temp,mean_max_temp,mean_temp,temp_variability
station,,,,
"El Loa, Calama Ad.",2.992877,24.009041,13.500959,21.016164
"Quinta Normal, Santiago",8.536438,23.550137,16.043288,15.013699
Pudahuel Santiago,8.192877,22.926575,15.559726,14.733699
"Eulógio Sánchez, Tobalaba Ad.",8.845753,22.845429,15.845591,13.999676
"General Freire, Curicó Ad.",7.877808,21.249863,14.563836,13.372055
"María Dolores, Los Angeles Ad.",7.113536,20.327778,13.720657,13.214242
"General Bernardo O'Higgins, Chillán Ad.",7.670685,20.392603,14.031644,12.721918
Chile Chico Ad.,3.228767,15.519780,9.374274,12.291013
"Maquehue, Temuco Ad.",6.345933,18.150962,12.248447,11.805029


In [91]:
top_compare = pd.concat([
    coldest.assign(group="Coldest"),
    warmest.assign(group="Warmest")
])

fig = px.bar(
    top_compare.reset_index(),
    x="station",
    y="mean_temp",
    color="group",
    title="Coldest vs Warmest Stations (Mean Temperature)"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

**Temperature ranking across stations**

Stations show clear differences in climate conditions:

*Coldest stations*

Southern and high-latitude stations consistently show the lowest average temperatures, reflecting strong geographic and latitudinal effects.

*Warmest stations*

Northern stations exhibit significantly higher mean temperatures, consistent with arid and subtropical climate zones.

*Most variable climates*

Stations with the largest temperature differences between minimum and maximum values tend to correspond to regions with:

- stronger diurnal variation
- more continental influence
- or less maritime moderation

Climate structure in Chile is strongly spatial and can be quantified using simple descriptive statistics alone.

## Combined Climate Summary Table

In [92]:
station_summary = df.groupby("station").agg(
    mean_min_temp=("min_temp", "mean"),
    mean_max_temp=("max_temp", "mean"),
)

station_summary["mean_temp"] = (
    station_summary["mean_min_temp"] + station_summary["mean_max_temp"]
) / 2

station_summary["temp_variability"] = (
    station_summary["mean_max_temp"] - station_summary["mean_min_temp"]
)

In [93]:
station_summary["cold_rank"] = station_summary["mean_temp"].rank(method="min", ascending=True)
station_summary["warm_rank"] = station_summary["mean_temp"].rank(method="min", ascending=False)
station_summary["variability_rank"] = station_summary["temp_variability"].rank(method="min", ascending=False)

In [96]:
summary_table = station_summary.reset_index()[[
    "station",
    "mean_temp",
    "temp_variability",
    "cold_rank",
    "warm_rank",
    "variability_rank"
]]

summary_table

,station,mean_temp,temp_variability,cold_rank,warm_rank,variability_rank
0,Alto Palena Ad.,10.466565,9.329230,12.0,22.0,17.0
1,Balmaceda Ad.,7.387123,9.928767,6.0,28.0,14.0
2,"C.M.A. Eduardo Frei Montalva, Antártica",-2.484110,3.616986,1.0,33.0,33.0
3,"Carlos Ibañez, Punta Arenas Ap.",6.406849,6.650959,3.0,31.0,26.0
4,"Carriel Sur, Concepción.",13.415616,8.942466,18.0,16.0,20.0
5,"Cañal Bajo, Osorno Ad.",11.261918,10.448767,14.0,20.0,11.0
6,Cerro Moreno Antofagasta Ap.,16.760548,5.046027,30.0,4.0,29.0
7,"Chacalluta, Arica Ap.",19.206044,4.543956,32.0,2.0,32.0
8,Chile Chico Ad.,9.374274,12.291013,9.0,25.0,8.0
9,"Desierto de Atacama, Caldera Ad.",15.590411,8.057534,27.0,7.0,22.0


**Unified climate ranking across stations**

This consolidated view allows direct comparison of stations across three key climate dimensions:

- Mean temperature (climate warmth/coldness)
- Temperature variability (climate stability)
- Relative ranking across both dimensions

**Key interpretation**

- Stations at the top of the cold rank correspond to consistently colder climates.
- Stations with high variability rank indicate stronger temperature swings between day and night or across seasons.
- Warm and cold rankings are inversely related, as expected, but variability is independent and reveals additional climate structure.

## Data Quality Overview

- Missing values: <1% (low)
- Duplicates: none
- Outliers: minimal and likely valid
- Station coverage: balanced
- Temporal coverage: full year (2014)

---

## **Potential Target Variable**

**Candidate target:**

- `max_temp`
- `min_temp`

**Potential ML task:**

- **Time series forecasting** (primary goal)
- **Regression** (baseline modeling approach)

**Rationale:**

Temperature is a continuous variable with strong seasonal structure and geographic variation. This makes it suitable for forecasting future values based on historical patterns.

Accurate predictions of temperature trends can then be translated into tourism insights, such as:

- identifying warmest and coldest travel periods,
- comparing seasonal comfort across regions,
- and supporting destination planning decisions.

---

# EDA Findings Summary

## **Key Findings**

- The dataset has **very low missing data (<1%)**, concentrated only in temperature variables, making it suitable for analysis without aggressive imputation.
- **Strong positive correlation (0.66)** exists between `min_temp` and `max_temp`, reflecting consistent daily climate structure.
- Temperature variables contain a small proportion of outliers (<1%), consistent with natural climate extremes.
- The data exhibits **clear seasonality**, indicating strong time-driven patterns in temperature behavior.
- Station data is **well-balanced geographically**, with only minor variations in record counts across locations.
- The dataset is well-suited for **regression and time series forecasting**, as target variables are continuous and structurally stable.

The dataset is high-quality and internally consistent, with strong seasonal and geographic structure. While linear correlations with time features are weak, deeper analysis reveals clear non-linear seasonal behavior, making it appropriate for forecasting-based modeling.

This dataset is best suited for:

- **Seasonal time series forecasting**
- **Climate pattern analysis**
- **Tourism-oriented decision support based on temperature trends**

---

# Appendix

### References

- Dataset source
- Internal course materials (Sonda Bootcamp, 2026)
- Machine Learning notebooks from the shared Assistanship Google Drive folder
- [Pandas Documentation](https://pandas.pydata.org/docs/user_guide/index.html)
- [Plotly Documentation](https://docs.plotly.com/)

### Acknowledgements
I would like to thank my instructors for their guidance, continuous support, and encouragement throughout this project.

I also acknowledge the use of AI-assisted tools for support with debugging, code review, documentation, and learning concepts during the development of the analysis. All final decisions, interpretations, and conclusions presented in this work remain my own.